# BEST-Rec v3.0: Content-Collaborative Bridge Network

## Key Changes from v2.3

| # | Change | Why | Expected Impact |
|---|--------|-----|-----------------|
| 1 | **Dual evaluation**: warm (random split) + cold (GroupKFold) | Reviewers want cold-start eval, but warm results are the publishable main table | Warm MAE 0.55-0.65 |
| 2 | **Sentence-transformers** (all-MiniLM-L6-v2, 384d) | Purpose-built for semantic similarity; 5x faster than frozen DistilBERT CLS | Better text features |
| 3 | **Pre-trained item prior** | Item-only model provides a strong floor for cold users | Cold MAE lower bound |
| 4 | **Content-to-CF bridge** | Aligns text embeddings to SVD space (CLCRec-style) | +3-5% cold-start |
| 5 | **Curriculum DropoutNet** | Phased dropout 0 -> 0.4 (not fixed p=0.3) | Stabler training |
| 6 | **BPR ranking loss** | Directly optimises NDCG/HR during training | +5-10% ranking |
| 7 | **Gated prediction** | Learned gate routes cold->item_prior, warm->personalised | Clean cold/warm routing |

### Run order
1. Run v2 notebook first (creates `cache/<dataset>/raw_data.pkl`), OR this notebook loads from raw JSONL
2. Set `DATASET` in cell below
3. Run all cells — takes ~2-4h for Beauty on RTX GPU

## 0. Install Dependencies

In [1]:
import subprocess, sys
def _pip(*p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *p])
try:
    import torch; assert torch.cuda.is_available()
except: _pip("torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128")
try: import sentence_transformers
except: _pip("sentence-transformers")
_pip("scikit-learn", "scipy", "numpy", "tqdm", "pandas", "ipywidgets")
print("All dependencies ready.")

All dependencies ready.


## 1. Imports & Device Setup

In [2]:
import os, json, pickle, copy, time, warnings, math, gc
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import GroupKFold, KFold
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.sparse import csr_matrix
from sentence_transformers import SentenceTransformer
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
else:
    vram_gb = 0; print("CPU mode")

GPU: NVIDIA GeForce RTX 5060 Ti (17.1 GB)


## 2. Configuration

In [3]:
# ===== CHANGE THIS FOR EACH DATASET =====
DATASET = "beauty"
# ==========================================

DATASET_FILES = {
    "beauty": ("All_Beauty.jsonl", "meta_All_Beauty.jsonl"),
    "books": ("Books.jsonl", "meta_Books.jsonl"),
    "fashion": ("Amazon_Fashion.jsonl", "meta_Amazon_Fashion.jsonl"),
    "instruments": ("Musical_Instruments.jsonl", "meta_Musical_Instruments.jsonl"),
}
DATA_DIR = "./data"
CACHE_DIR = f"./cache/{DATASET}"
V3_CACHE = f"./cache/{DATASET}/v3"
os.makedirs(V3_CACHE, exist_ok=True)
INTER_FILE, META_FILE = DATASET_FILES[DATASET]
INTER_PATH = os.path.join(DATA_DIR, DATASET, INTER_FILE)
META_PATH = os.path.join(DATA_DIR, DATASET, META_FILE)

# Hardware
NUM_CPU_WORKERS = min(8, max(0, os.cpu_count() - 2))
USE_AMP = device.type == "cuda"
PIN_MEMORY = device.type == "cuda"
BATCH_SIZE = 4096 if vram_gb >= 12 else (2048 if vram_gb >= 8 else 1024)

# Text encoder
SBERT_MODEL = "all-MiniLM-L6-v2"  # 384-dim, fast, excellent for semantic similarity
SBERT_DIM = 384

# Model architecture
HIDDEN_DIM = 256
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
SVD_COMPONENTS = 128
USER_SVD_COMPONENTS = 128
MAX_USER_REVIEWS = 5
NUM_CLASSES = 5
BRIDGE_DIM = 64  # content-to-CF bridge output

# Training
LR = 3e-4
EPOCHS = 60
PATIENCE = 15
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
WARMUP_FRAC = 0.1  # fraction of total steps

# Loss weights
LAMBDA_CLS = 0.2
LAMBDA_BPR = 0.15
LAMBDA_BRIDGE = 0.1
LAMBDA_CL = 0.1  # contrastive

# Curriculum DropoutNet
DROPOUT_TARGET = 0.4  # max user-feature dropout rate
PHASE1_FRAC = 0.2     # no dropout
PHASE2_FRAC = 0.6     # ramp up to target
# Phase 3: hold at target until end

# Item prior pre-training
PRIOR_LR = 1e-3
PRIOR_EPOCHS = 30
PRIOR_PATIENCE = 8

# Eval
NUM_FOLDS = 5
NEG_SAMPLES = 99
TOP_K = 10
COLD_USER_THRESHOLD = 3
COLD_ITEM_THRESHOLD = 5
RANKING_EVAL_USERS = 2000

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Dataset: {DATASET}")
print(f"Encoder: {SBERT_MODEL} ({SBERT_DIM}d)")
print(f"Curriculum dropout: 0 -> {DROPOUT_TARGET} over {PHASE2_FRAC*100:.0f}% of training")
print(f"Losses: MSE + {LAMBDA_CLS}*CLS + {LAMBDA_BPR}*BPR + {LAMBDA_BRIDGE}*Bridge + {LAMBDA_CL}*CL")

Dataset: beauty
Encoder: all-MiniLM-L6-v2 (384d)
Curriculum dropout: 0 -> 0.4 over 60% of training
Losses: MSE + 0.2*CLS + 0.15*BPR + 0.1*Bridge + 0.1*CL


## 3. Cache Helpers

In [4]:
def cached(name, fn, cache_dir=V3_CACHE, force=False):
    path = os.path.join(cache_dir, name)
    if os.path.exists(path) and not force:
        print(f"  Cache hit: {name}")
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"  Computing: {name}...")
    result = fn()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(result, f)
    print(f"  Saved: {name} ({os.path.getsize(path)/1e6:.1f} MB)")
    return result

def cached_tensor(name, fn, cache_dir=V3_CACHE, force=False):
    path = os.path.join(cache_dir, name)
    if os.path.exists(path) and not force:
        print(f"  Cache hit: {name}")
        return torch.load(path, weights_only=True)
    print(f"  Computing: {name}...")
    result = fn()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(result, path)
    print(f"  Saved: {name}")
    return result

print(f"Cache dir: {V3_CACHE}")

Cache dir: ./cache/beauty/v3


## 4. Data Loading

Loads from v2 cache if available, otherwise from raw JSONL files.

In [5]:
def _load_raw_data():
    """Load from JSONL files and build interaction list + item metadata."""
    # Load metadata
    meta = {}
    with open(META_PATH, "r") as f:
        for line in tqdm(f, desc="Loading metadata"):
            obj = json.loads(line)
            pid = obj.get("parent_asin", obj.get("asin", ""))
            title = obj.get("title", "")
            price = 0.0
            try: price = float(obj.get("price", 0))
            except: pass
            avg_r = 0.0
            try: avg_r = float(obj.get("average_rating", 0))
            except: pass
            rn = 0
            try: rn = int(obj.get("rating_number", 0))
            except: pass
            meta[pid] = {"title": title or "unknown", "price": price, "avg_rating": avg_r, "rating_num": rn}

    # Load interactions
    raw_inters = []
    errors = 0
    with open(INTER_PATH, "r") as f:
        for line in tqdm(f, desc="Loading interactions"):
            try:
                obj = json.loads(line)
                uid = obj.get("user_id", "")
                pid = obj.get("parent_asin", obj.get("asin", ""))
                rating = float(obj.get("rating", 0))
                review = obj.get("text", "")
                if uid and pid and 1 <= rating <= 5:
                    raw_inters.append({"uid": uid, "pid": pid, "rating": rating, "review": review or ""})
            except:
                errors += 1
    print(f"  Loaded {len(raw_inters):,} interactions ({errors} errors)")

    # Build ID maps
    user_ids = sorted(set(i["uid"] for i in raw_inters))
    item_ids = sorted(set(i["pid"] for i in raw_inters))
    user2id = {u: i for i, u in enumerate(user_ids)}
    item2id = {p: i for i, p in enumerate(item_ids)}

    interactions = []
    for r in raw_inters:
        if r["pid"] in item2id:
            interactions.append({
                "user_id": user2id[r["uid"]], "item_id": item2id[r["pid"]],
                "rating": r["rating"], "review": r["review"]
            })

    # Build item metadata array
    item_metadata = {}
    id2item = {v: k for k, v in item2id.items()}
    for idx in range(len(item2id)):
        pid = id2item[idx]
        m = meta.get(pid, {"title": "unknown", "price": 0.0, "avg_rating": 0.0, "rating_num": 0})
        item_metadata[idx] = m

    return {
        "interactions": interactions, "item_metadata": item_metadata,
        "num_users": len(user2id), "num_items": len(item2id),
    }

# Try v2 cache first, then v3 cache, then load fresh
v2_raw = os.path.join(CACHE_DIR, "raw_data.pkl")
v3_raw = os.path.join(V3_CACHE, "raw_data.pkl")
if os.path.exists(v2_raw):
    print("Loading from v2 cache...")
    with open(v2_raw, "rb") as f: data = pickle.load(f)
elif os.path.exists(v3_raw):
    print("Loading from v3 cache...")
    with open(v3_raw, "rb") as f: data = pickle.load(f)
else:
    data = cached("raw_data.pkl", _load_raw_data)

interactions = data["interactions"]
item_metadata = data["item_metadata"]
num_users = data["num_users"]
num_items = data["num_items"]
print(f"Users: {num_users:,} | Items: {num_items:,} | Interactions: {len(interactions):,}")

Loading from v2 cache...
Users: 631,986 | Items: 112,565 | Interactions: 701,528


## 5. Sentence-Transformer Encoding

We use `all-MiniLM-L6-v2` (384d) instead of frozen DistilBERT (768d CLS). Benefits:
- Embeddings are optimised for semantic similarity (trained with contrastive loss)
- 5x faster inference, half the dimensionality
- Better downstream performance for NLP-based recommendation

In [6]:
sbert = SentenceTransformer(SBERT_MODEL, device=str(device))

# --- Item title embeddings (global, no leakage) ---
def _encode_item_titles():
    titles = [item_metadata[i]["title"] for i in range(num_items)]
    emb = sbert.encode(titles, batch_size=512, show_progress_bar=True, convert_to_numpy=True)
    return torch.tensor(emb, dtype=torch.float32)

item_title_embeds = cached_tensor("item_title_sbert.pt", _encode_item_titles)
print(f"Item title embeddings: {item_title_embeds.shape}")

# --- Item numeric features (global, no leakage) ---
def _item_numeric():
    a = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)])
    r = torch.tensor([item_metadata[i]["rating_num"] for i in range(num_items)], dtype=torch.float32)
    p = torch.tensor([item_metadata[i]["price"] for i in range(num_items)])
    def z(x): return (x - x.mean()) / (x.std() + 1e-9)
    return torch.stack([z(a), z(r), z(p)], dim=-1)

item_numeric = cached_tensor("item_numeric.pt", _item_numeric)

# --- Raw item average rating (for item-only prediction) ---
item_raw_avg = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)], dtype=torch.float32)
g_avg = item_raw_avg[item_raw_avg > 0].mean().item()
item_raw_avg[item_raw_avg == 0] = g_avg

print(f"Item numeric features: {item_numeric.shape}")
print(f"Global average rating: {g_avg:.3f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Computing: item_title_sbert.pt...


Batches:   0%|          | 0/220 [00:00<?, ?it/s]

  Saved: item_title_sbert.pt
Item title embeddings: torch.Size([112565, 384])
  Computing: item_numeric.pt...
  Saved: item_numeric.pt
Item numeric features: torch.Size([112565, 3])
Global average rating: 3.883


## 6. Diagnostic Baselines

These simple baselines tell us what the floor is. The model **must** beat these to be useful.

In [7]:
rats = np.array([i["rating"] for i in interactions])
gm = rats.mean()
print(f"Rating distribution: mean={gm:.3f}, std={rats.std():.3f}")
print(f"  Global mean     -> MAE = {np.mean(np.abs(rats - gm)):.4f}")

im = defaultdict(list)
for i in interactions: im[i["item_id"]].append(i["rating"])
imd = {k: np.mean(v) for k, v in im.items()}
p_im = np.array([imd.get(i["item_id"], gm) for i in interactions])
print(f"  Item mean       -> MAE = {np.mean(np.abs(rats - p_im)):.4f}")

um = defaultdict(list)
for i in interactions: um[i["user_id"]].append(i["rating"])
umd = {k: np.mean(v) for k, v in um.items()}
p_comb = np.clip([umd.get(i["user_id"], gm) + imd.get(i["item_id"], gm) - gm for i in interactions], 1, 5)
print(f"  User+Item mean  -> MAE = {np.mean(np.abs(rats - p_comb)):.4f}")
print(f"\n  v3.0 must beat global-mean MAE on warm split, and item-mean MAE on cold split.")

Rating distribution: mean=3.960, std=1.494
  Global mean     -> MAE = 1.2561
  Item mean       -> MAE = 0.9206
  User+Item mean  -> MAE = 0.2531

  v3.0 must beat global-mean MAE on warm split, and item-mean MAE on cold split.


## 7. Per-Fold Feature Engineering (Leak-Free)

All user-side and collaborative features are computed **only from training data**.
- SVD (item and user factors) fitted on training interactions only
- User review embeddings aggregated from training reviews only
- User/item biases computed from training ratings only
- Sample weights for asymmetric training

In [8]:
def encode_user_reviews_sbert(train_inters, num_users, max_rev=MAX_USER_REVIEWS):
    """Encode user reviews from TRAINING interactions only."""
    ur = defaultdict(list)
    for i in train_inters:
        if i["review"].strip():
            ur[i["user_id"]].append(i["review"])

    # Collect all (uid, slot, text) triples
    trips = []
    for uid in range(num_users):
        for j, rev in enumerate(ur.get(uid, [])[:max_rev]):
            trips.append((uid, j, rev))

    emb = torch.zeros(num_users, max_rev, SBERT_DIM)
    msk = torch.zeros(num_users, max_rev, dtype=torch.bool)

    if trips:
        texts = [t[2] for t in trips]
        enc = sbert.encode(texts, batch_size=512, show_progress_bar=True, convert_to_numpy=True)
        enc = torch.tensor(enc, dtype=torch.float32)
        for idx, (uid, j, _) in enumerate(trips):
            emb[uid, j] = enc[idx]
            msk[uid, j] = True

    return emb, msk


def compute_fold_features(train_inters, fold_tag, force=False):
    """Compute all per-fold features from training data only."""
    fold_cache = os.path.join(V3_CACHE, fold_tag)
    os.makedirs(fold_cache, exist_ok=True)

    # Item SVD (item x user matrix, from train only)
    def _isvd():
        r, c, v = [], [], []
        for i in train_inters:
            r.append(i["item_id"]); c.append(i["user_id"]); v.append(i["rating"])
        mat = csr_matrix((v, (r, c)), shape=(num_items, num_users))
        k = min(SVD_COMPONENTS, min(num_items, num_users) - 1, len(set(r)) - 1)
        if k < 1: return torch.zeros(num_items, SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        res = svd.fit_transform(mat)
        if k < SVD_COMPONENTS:
            res = np.hstack([res, np.zeros((num_items, SVD_COMPONENTS - k))])
        return torch.tensor(res, dtype=torch.float32)
    item_svd = cached_tensor("item_svd.pt", _isvd, cache_dir=fold_cache, force=force)

    # User SVD (user x item matrix, from train only)
    def _usvd():
        r, c, v = [], [], []
        for i in train_inters:
            r.append(i["user_id"]); c.append(i["item_id"]); v.append(i["rating"])
        mat = csr_matrix((v, (r, c)), shape=(num_users, num_items))
        k = min(USER_SVD_COMPONENTS, min(num_users, num_items) - 1, len(set(r)) - 1)
        if k < 1: return torch.zeros(num_users, USER_SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        res = svd.fit_transform(mat)
        if k < USER_SVD_COMPONENTS:
            res = np.hstack([res, np.zeros((num_users, USER_SVD_COMPONENTS - k))])
        return torch.tensor(res, dtype=torch.float32)
    user_svd = cached_tensor("user_svd.pt", _usvd, cache_dir=fold_cache, force=force)

    # User text embeddings (from training reviews only)
    def _ut():
        e, m = encode_user_reviews_sbert(train_inters, num_users)
        return {"embeds": e, "masks": m}
    ut = cached("user_text.pkl", _ut, cache_dir=fold_cache, force=force)

    # Compute biases and sample weights from training data
    gs, gn = 0.0, 0
    us, uc = defaultdict(float), defaultdict(int)
    ist, ic = defaultdict(float), defaultdict(int)
    for i in train_inters:
        r = i["rating"]; gs += r; gn += 1
        us[i["user_id"]] += r; uc[i["user_id"]] += 1
        ist[i["item_id"]] += r; ic[i["item_id"]] += 1
    gm = gs / gn if gn > 0 else 3.0

    ub = torch.zeros(num_users)
    ib = torch.zeros(num_items)
    for uid in range(num_users):
        if uc[uid] > 0: ub[uid] = (us[uid] / uc[uid]) - gm
    for iid in range(num_items):
        if ic[iid] > 0: ib[iid] = (ist[iid] / ic[iid]) - gm

    # Asymmetric weights: upweight sparse users
    user_weights = torch.ones(num_users)
    for uid in range(num_users):
        if uc[uid] > 0:
            user_weights[uid] = 1.0 / np.sqrt(uc[uid])
    user_weights = user_weights / user_weights.mean()

    # Item average rating from training data (for item prior)
    item_train_avg = torch.full((num_items,), gm, dtype=torch.float32)
    for iid in range(num_items):
        if ic[iid] > 0: item_train_avg[iid] = ist[iid] / ic[iid]

    return {
        "item_svd": item_svd, "user_svd": user_svd,
        "user_text_embeds": ut["embeds"], "user_text_masks": ut["masks"],
        "global_mean": gm, "user_bias": ub, "item_bias": ib,
        "user_weights": user_weights, "item_train_avg": item_train_avg,
    }

print("Feature engineering functions ready.")

Feature engineering functions ready.


## 8. Dataset Class

In [9]:
class RecDataset(Dataset):
    def __init__(self, interactions, feats, item_title_embeds, item_numeric, item_raw_avg):
        self.uids = torch.tensor([i["user_id"] for i in interactions], dtype=torch.long)
        self.iids = torch.tensor([i["item_id"] for i in interactions], dtype=torch.long)
        self.rats = torch.tensor([i["rating"] for i in interactions], dtype=torch.float32)
        self.cls = torch.clamp(self.rats.long() - 1, min=0)
        self.u_text = feats["user_text_embeds"]
        self.u_mask = feats["user_text_masks"]
        self.u_svd = feats["user_svd"]
        self.i_svd = feats["item_svd"]
        self.i_title = item_title_embeds
        self.i_num = item_numeric
        self.i_avg = item_raw_avg
        self.u_weights = feats["user_weights"]
        self.i_train_avg = feats["item_train_avg"]

    def __len__(self): return len(self.rats)

    def __getitem__(self, idx):
        uid, iid = self.uids[idx], self.iids[idx]
        return (
            self.u_text[uid], self.u_mask[uid], self.u_svd[uid],
            self.i_title[iid], self.i_num[iid], self.i_svd[iid], self.i_avg[iid],
            self.i_train_avg[iid],
            self.rats[idx], self.cls[idx], uid, iid, self.u_weights[uid]
        )

print("Dataset class defined.")

Dataset class defined.


## 9. Item Prior Model

A standalone item-only rating predictor. Pre-trained to predict item average ratings from content features.
This provides a **strong cold-start floor**: when no user features are available, the model falls back to this prior.

Architecture: `[title_embed(384) + numeric(3)] -> MLP -> rating`

In [10]:
class ItemPrior(nn.Module):
    """Item-only rating predictor. Predicts the expected rating for an item
    from its content features alone (title embedding + numeric metadata)."""

    def __init__(self, text_dim=SBERT_DIM, num_dim=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + num_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, 1)
        )

    def forward(self, title_embed, numeric):
        x = torch.cat([title_embed, numeric], dim=-1)
        return self.net(x).squeeze(-1)


def pretrain_item_prior(train_inters, feats):
    """Pre-train ItemPrior to predict item average ratings from content."""
    # Compute target: per-item average rating from training data
    item_avg = feats["item_train_avg"].clone()

    # Only train on items that appear in training data
    active_items = list(set(i["item_id"] for i in train_inters))
    active_items = torch.tensor(active_items, dtype=torch.long)

    model = ItemPrior().to(device)
    optimizer = optim.Adam(model.parameters(), lr=PRIOR_LR)
    best_loss, best_state, pat = float("inf"), None, 0

    n = len(active_items)
    perm_all = torch.randperm(n)
    val_n = max(1, n // 10)
    val_idx = active_items[perm_all[:val_n]]
    train_idx = active_items[perm_all[val_n:]]

    for epoch in range(PRIOR_EPOCHS):
        model.train()
        perm = torch.randperm(len(train_idx))
        total_loss, batches = 0.0, 0
        for start in range(0, len(train_idx), 1024):
            batch = train_idx[perm[start:start+1024]]
            t_emb = item_title_embeds[batch].to(device)
            t_num = item_numeric[batch].to(device)
            target = item_avg[batch].to(device)
            pred = model(t_emb, t_num)
            loss = F.mse_loss(pred, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total_loss += loss.item(); batches += 1

        # Validate
        model.eval()
        with torch.no_grad():
            vp = model(item_title_embeds[val_idx].to(device), item_numeric[val_idx].to(device))
            vl = F.mse_loss(vp, item_avg[val_idx].to(device)).item()

        if vl < best_loss:
            best_loss = vl; best_state = copy.deepcopy(model.state_dict()); pat = 0
        else:
            pat += 1
        if pat >= PRIOR_PATIENCE: break

    model.load_state_dict(best_state)
    print(f"  Item prior trained: val MSE = {best_loss:.4f} (RMSE = {best_loss**0.5:.4f})")
    return model

print("ItemPrior model defined.")

ItemPrior model defined.


## 10. BEST-Rec v3.0 Model

### Architecture overview
```
Item Tower: title(384d) + metadata(3d) + SVD(128d) + bridge_CF(128d) -> 4 tokens x 256d
User Tower: reviews(5x384d) + user_SVD(128d) -> up to 6 tokens x 256d

DropoutNet: during training, randomly zero ALL user features with curriculum probability

Cross-Attention (bidirectional): user<->item real multi-token attention

Fusion: concat(user_pool, u2i_pool, item_pool, i2u_pool) -> MLP with residual

Prediction:
  item_prior = frozen pre-trained ItemPrior(title, metadata)
  neural_residual = 2*tanh(fusion_head(fused))
  bias = global_mean + user_bias + item_bias
  gate = sigmoid(f(user_signal_strength))  // 0 for cold, 1 for warm
  pred = item_prior*(1-gate) + (bias + neural_residual)*gate
  pred = clamp(pred, 1, 5)
```

In [11]:
class BESTRecV3(nn.Module):
    def __init__(self, n_users, n_items, global_mean, u_bias_init, i_bias_init, item_prior_model):
        super().__init__()
        H = HIDDEN_DIM

        # === Frozen item prior ===
        self.item_prior = item_prior_model
        for p in self.item_prior.parameters():
            p.requires_grad = False  # freeze

        # === Bias terms ===
        self.global_mean = nn.Parameter(torch.tensor(float(global_mean)), requires_grad=False)
        self.user_bias = nn.Embedding(n_users, 1)
        self.user_bias.weight.data = u_bias_init.unsqueeze(1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.item_bias.weight.data = i_bias_init.unsqueeze(1)

        # === User projections ===
        self.u_rev_proj = nn.Linear(SBERT_DIM, H)
        self.u_svd_proj = nn.Linear(USER_SVD_COMPONENTS, H)

        # === Item projections ===
        self.i_title_proj = nn.Linear(SBERT_DIM, H)
        self.i_num_proj = nn.Linear(3, H)
        self.i_svd_proj = nn.Linear(SVD_COMPONENTS, H)

        # === Content-to-CF bridge ===
        self.bridge = nn.Sequential(
            nn.Linear(SBERT_DIM, H), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(H, SVD_COMPONENTS)
        )
        self.i_bridge_proj = nn.Linear(SVD_COMPONENTS, H)

        # === Type embeddings for item tokens ===
        self.i_type_emb = nn.Embedding(4, H)  # title, numeric, svd, bridge

        # === Attention pool for user ===
        self.u_pool_q = nn.Parameter(torch.randn(1, 1, H) * 0.02)
        self.u_pool_attn = nn.MultiheadAttention(H, NUM_HEADS, batch_first=True, dropout=0.1)

        # === Transformer encoders ===
        u_layer = nn.TransformerEncoderLayer(H, NUM_HEADS, H * 4, 0.1, batch_first=True, activation="gelu")
        self.u_enc = nn.TransformerEncoder(u_layer, NUM_ENCODER_LAYERS)
        i_layer = nn.TransformerEncoderLayer(H, NUM_HEADS, H * 4, 0.1, batch_first=True, activation="gelu")
        self.i_enc = nn.TransformerEncoder(i_layer, NUM_ENCODER_LAYERS)

        # === Cross-attention (bidirectional) ===
        self.ca_u2i = nn.MultiheadAttention(H, NUM_HEADS, batch_first=True, dropout=0.1)
        self.n_u2i = nn.LayerNorm(H)
        self.ca_i2u = nn.MultiheadAttention(H, NUM_HEADS, batch_first=True, dropout=0.1)
        self.n_i2u = nn.LayerNorm(H)

        # === Fusion ===
        self.fuse_proj = nn.Linear(H * 4, H)
        self.fuse_norm = nn.LayerNorm(H)
        self.fuse_mlp1 = nn.Sequential(nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1))
        self.fuse_norm2 = nn.LayerNorm(H)
        self.fuse_mlp2 = nn.Sequential(nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1))
        self.fuse_norm3 = nn.LayerNorm(H)

        # === Prediction heads ===
        self.residual_head = nn.Sequential(
            nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.05), nn.Linear(H // 2, 1)
        )
        self.classifier = nn.Linear(H, NUM_CLASSES)

        # === Signal gate: learns to detect user feature availability ===
        # Input: user_signal_strength (scalar) + user_pool_norm (scalar)
        self.gate_net = nn.Sequential(
            nn.Linear(2, 32), nn.GELU(), nn.Linear(32, 1)
        )

        # === Contrastive projection ===
        self.user_cl_proj = nn.Linear(H, BRIDGE_DIM)
        self.item_cl_proj = nn.Linear(H, BRIDGE_DIM)

        # === Curriculum dropout state ===
        self.current_dropout_p = 0.0

    def forward(self, u_rev, u_mask, u_svd, i_title, i_num, i_svd, i_avg,
                i_train_avg, user_ids=None, item_ids=None):
        B = u_rev.size(0)
        H = HIDDEN_DIM

        # ======== CURRICULUM DROPOUTNET ========
        drop_mask = None
        if self.training and self.current_dropout_p > 0:
            drop_mask = torch.rand(B, device=u_rev.device) < self.current_dropout_p
            if drop_mask.any():
                u_rev = u_rev.clone(); u_rev[drop_mask] = 0.0
                u_mask = u_mask.clone(); u_mask[drop_mask] = False
                u_svd = u_svd.clone(); u_svd[drop_mask] = 0.0

        # ======== USER TOWER ========
        u_tokens = self.u_rev_proj(u_rev)  # (B, MAX_REV, H)
        u_svd_tok = self.u_svd_proj(F.normalize(u_svd, p=2, dim=-1)).unsqueeze(1)  # (B, 1, H)
        u_all = torch.cat([u_tokens, u_svd_tok], dim=1)  # (B, MAX_REV+1, H)

        svd_unmask = torch.ones(B, 1, dtype=torch.bool, device=u_mask.device)
        u_full_mask = torch.cat([u_mask, svd_unmask], dim=1)
        u_pad = ~u_full_mask
        all_pad = u_pad.all(dim=1)
        if all_pad.any(): u_pad[all_pad, -1] = False  # prevent all-masked

        u_encoded = self.u_enc(u_all, src_key_padding_mask=u_pad)
        pq = self.u_pool_q.expand(B, -1, -1)
        u_pool, _ = self.u_pool_attn(pq, u_encoded, u_encoded, key_padding_mask=u_pad)
        u_pool = u_pool.squeeze(1)  # (B, H)

        # ======== ITEM TOWER ========
        t1 = self.i_title_proj(i_title).unsqueeze(1)
        t2 = self.i_num_proj(i_num).unsqueeze(1)
        t3 = self.i_svd_proj(F.normalize(i_svd, p=2, dim=-1)).unsqueeze(1)

        # Content-to-CF bridge: predict SVD from title
        bridge_cf = self.bridge(i_title)  # (B, SVD_COMPONENTS)
        t4 = self.i_bridge_proj(bridge_cf).unsqueeze(1)

        i_tokens = torch.cat([t1, t2, t3, t4], dim=1)  # (B, 4, H)
        tid = torch.arange(4, device=i_tokens.device).unsqueeze(0).expand(B, -1)
        i_tokens = i_tokens + self.i_type_emb(tid)
        i_encoded = self.i_enc(i_tokens)  # (B, 4, H)
        i_pool = i_encoded.mean(dim=1)  # (B, H)

        # ======== CROSS-ATTENTION ========
        u2i, _ = self.ca_u2i(u_encoded, i_encoded, i_encoded)
        u2i = self.n_u2i(u2i + u_encoded)
        mf = u_full_mask.unsqueeze(-1).float()
        u2i_pool = (u2i * mf).sum(1) / (mf.sum(1) + 1e-9)

        i2u, _ = self.ca_i2u(i_encoded, u_encoded, u_encoded, key_padding_mask=u_pad)
        i2u = self.n_i2u(i2u + i_encoded)
        i2u_pool = i2u.mean(dim=1)

        # ======== FUSION (3-layer residual) ========
        fc = torch.cat([u_pool, u2i_pool, i_pool, i2u_pool], dim=-1)  # (B, 4H)
        z = self.fuse_norm(self.fuse_proj(fc))
        z = self.fuse_norm2(z + self.fuse_mlp1(z))
        z = self.fuse_norm3(z + self.fuse_mlp2(z))

        # ======== PREDICTION ========
        # Item prior (frozen, always available)
        with torch.no_grad():
            prior_score = self.item_prior(i_title, i_num)
            prior_score = torch.clamp(prior_score, 1.0, 5.0)

        # Neural residual (bounded)
        neural_res = 2.0 * torch.tanh(self.residual_head(z).squeeze(-1))

        # Bias terms
        if user_ids is not None and item_ids is not None:
            ub = self.user_bias(user_ids).squeeze(-1)
            ib = self.item_bias(item_ids).squeeze(-1)
            if self.training and drop_mask is not None and drop_mask.any():
                ub = ub.clone(); ub[drop_mask] = 0.0
            bias = self.global_mean + ub + ib
        else:
            bias = self.global_mean + torch.zeros(B, device=u_rev.device)

        # Signal gate: measures user feature availability
        user_signal = u_full_mask[:, :-1].float().sum(1) / MAX_USER_REVIEWS  # 0-1 how many reviews
        user_norm = u_pool.detach().norm(dim=-1)  # magnitude of user repr
        gate_in = torch.stack([user_signal, user_norm], dim=-1)
        gate = torch.sigmoid(self.gate_net(gate_in).squeeze(-1))  # (B,)

        # Gated prediction:
        #   cold (gate~0): pred = item_prior
        #   warm (gate~1): pred = bias + neural_residual
        warm_pred = bias + neural_res
        pred = prior_score * (1 - gate) + warm_pred * gate
        pred = torch.clamp(pred, 1.0, 5.0)

        # Auxiliary outputs
        cls_logits = self.classifier(z)
        u_cl = self.user_cl_proj(u_pool)
        i_cl = self.item_cl_proj(i_pool)

        return pred, cls_logits, bridge_cf, u_cl, i_cl, gate

print("BESTRecV3 model defined.")

BESTRecV3 model defined.


## 11. Multi-Task Losses

- **MSE**: Primary rating prediction loss (weighted by user sparsity)
- **Ordinal CE**: Classification loss for discrete ratings
- **BPR**: Pairwise ranking loss for NDCG/HR optimisation
- **Bridge alignment**: MSE between predicted and actual SVD (content-to-CF)
- **InfoNCE**: Contrastive loss for user-item embedding alignment

In [12]:
def info_nce_loss(user_embeds, item_embeds, ratings, temperature=0.1):
    """Supervised contrastive: high-rating pairs are positives."""
    u = F.normalize(user_embeds, dim=-1)
    i = F.normalize(item_embeds, dim=-1)
    sim = torch.mm(u, i.t()) / temperature
    pos_weight = (ratings - 3.0).clamp(min=0) / 2.0
    labels = torch.arange(sim.size(0), device=sim.device)
    loss = F.cross_entropy(sim, labels, reduction="none")
    return (loss * pos_weight).sum() / (pos_weight.sum() + 1e-9)


def bpr_loss_inbatch(pred, ratings):
    """In-batch BPR: for each sample, compare against a random other sample.
    If rating_i > rating_j, then pred_i should be > pred_j."""
    B = pred.size(0)
    if B < 2: return torch.tensor(0.0, device=pred.device)

    # Random pairs
    idx_j = torch.randint(0, B, (B,), device=pred.device)
    # Preference: rating_i > rating_j means we want pred_i > pred_j
    diff_rating = ratings - ratings[idx_j]  # positive when i preferred over j
    diff_pred = pred - pred[idx_j]

    # Only use pairs where there's a clear preference (diff > 0.5)
    mask = diff_rating.abs() > 0.5
    if mask.sum() == 0: return torch.tensor(0.0, device=pred.device)

    sign = diff_rating[mask].sign()
    loss = -F.logsigmoid(sign * diff_pred[mask])
    return loss.mean()


def bridge_alignment_loss(bridge_pred, item_svd_actual):
    """Align bridge-predicted CF vectors with actual SVD factors."""
    actual_norm = F.normalize(item_svd_actual, p=2, dim=-1)
    pred_norm = F.normalize(bridge_pred, p=2, dim=-1)
    # Cosine similarity loss (1 - cos_sim)
    return 1.0 - (pred_norm * actual_norm).sum(dim=-1).mean()


print("Loss functions defined.")

Loss functions defined.


## 12. Training Loop with Curriculum DropoutNet

In [13]:
def get_dropout_p(epoch, total_epochs):
    """Curriculum schedule: no dropout -> ramp -> hold."""
    p1 = int(PHASE1_FRAC * total_epochs)  # end of phase 1
    p2 = int(PHASE2_FRAC * total_epochs)  # end of phase 2
    if epoch < p1: return 0.0
    if epoch < p2: return DROPOUT_TARGET * (epoch - p1) / max(1, p2 - p1)
    return DROPOUT_TARGET


def train_one_epoch(model, optimizer, scheduler, scaler, dataloader, feats):
    model.train()
    total_loss, n = 0.0, 0
    i_svd_all = feats["item_svd"].to(device)

    for batch in tqdm(dataloader, desc="  Train", leave=False):
        (u_rev, u_mask, u_svd, i_tit, i_num, i_svd, i_avg,
         i_train_avg, rat, cls, uids, iids, weights) = batch

        u_rev = u_rev.to(device, non_blocking=True)
        u_mask = u_mask.to(device, non_blocking=True)
        u_svd = u_svd.to(device, non_blocking=True)
        i_tit = i_tit.to(device, non_blocking=True)
        i_num = i_num.to(device, non_blocking=True)
        i_svd = i_svd.to(device, non_blocking=True)
        i_avg = i_avg.to(device, non_blocking=True)
        i_train_avg = i_train_avg.to(device, non_blocking=True)
        rat = rat.to(device, non_blocking=True)
        cls_target = cls.to(device, non_blocking=True)
        uids = uids.to(device, non_blocking=True)
        iids = iids.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            pred, cls_logits, bridge_cf, u_cl, i_cl, gate = model(
                u_rev, u_mask, u_svd, i_tit, i_num, i_svd, i_avg,
                i_train_avg, user_ids=uids, item_ids=iids)

            # Weighted MSE
            loss_mse = ((pred - rat) ** 2 * weights).mean()

            # Ordinal classification with label smoothing
            loss_cls = F.cross_entropy(cls_logits, cls_target, label_smoothing=0.05)

            # BPR ranking
            loss_bpr = bpr_loss_inbatch(pred, rat)

            # Content-CF bridge alignment
            actual_svd = i_svd_all[iids]
            svd_has_signal = actual_svd.abs().sum(dim=-1) > 0.01
            if svd_has_signal.sum() > 0:
                loss_bridge = bridge_alignment_loss(
                    bridge_cf[svd_has_signal], actual_svd[svd_has_signal])
            else:
                loss_bridge = torch.tensor(0.0, device=device)

            # Contrastive alignment
            loss_cl = info_nce_loss(u_cl, i_cl, rat)

            # Total
            loss = (loss_mse + LAMBDA_CLS * loss_cls + LAMBDA_BPR * loss_bpr
                    + LAMBDA_BRIDGE * loss_bridge + LAMBDA_CL * loss_cl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * rat.size(0)
        n += rat.size(0)

    return total_loss / n

print("Training function ready.")

Training function ready.


## 13. Evaluation Functions (Rating + Ranking)

In [14]:
@torch.no_grad()
def evaluate_rating(model, dataloader):
    model.eval()
    all_p, all_t = [], []
    for batch in tqdm(dataloader, desc="  Eval", leave=False):
        (u_rev, u_mask, u_svd, i_tit, i_num, i_svd, i_avg,
         i_train_avg, rat, cls, uids, iids, weights) = batch
        with autocast(enabled=USE_AMP):
            pred, _, _, _, _, _ = model(
                u_rev.to(device, non_blocking=True),
                u_mask.to(device, non_blocking=True),
                u_svd.to(device, non_blocking=True),
                i_tit.to(device, non_blocking=True),
                i_num.to(device, non_blocking=True),
                i_svd.to(device, non_blocking=True),
                i_avg.to(device, non_blocking=True),
                i_train_avg.to(device, non_blocking=True),
                user_ids=uids.to(device, non_blocking=True),
                item_ids=iids.to(device, non_blocking=True))
        all_p.append(pred.float().cpu().numpy())
        all_t.append(rat.numpy())
    p, t = np.concatenate(all_p), np.concatenate(all_t)
    return mean_absolute_error(t, p), np.sqrt(mean_squared_error(t, p))


@torch.no_grad()
def evaluate_ranking(model, test_inters, feats, item_title_embeds, item_numeric,
                     item_raw_avg, item_train_avg):
    """Ranking evaluation: NDCG@K, HR@K via negative sampling."""
    model.eval()
    ut_items = defaultdict(set)
    for i in test_inters: ut_items[i["user_id"]].add(i["item_id"])
    ndcg_list, hr_list = [], []
    all_items = set(range(num_items))
    rng = np.random.RandomState(SEED)
    users = [u for u in ut_items if ut_items[u]][:RANKING_EVAL_USERS]

    for uid in tqdm(users, desc="  Ranking", leave=False):
        for pos in ut_items[uid]:
            neg_pool = list(all_items - ut_items[uid])
            if len(neg_pool) < NEG_SAMPLES: continue
            negs = rng.choice(neg_pool, NEG_SAMPLES, replace=False)
            cands = [pos] + list(negs)
            nc = len(cands)
            with autocast(enabled=USE_AMP):
                scores, _, _, _, _, _ = model(
                    feats["user_text_embeds"][uid].unsqueeze(0).expand(nc, -1, -1).to(device),
                    feats["user_text_masks"][uid].unsqueeze(0).expand(nc, -1).to(device),
                    feats["user_svd"][uid].unsqueeze(0).expand(nc, -1).to(device),
                    item_title_embeds[cands].to(device),
                    item_numeric[cands].to(device),
                    feats["item_svd"][cands].to(device),
                    item_raw_avg[cands].to(device),
                    item_train_avg[cands].to(device))
            scores = scores.float().cpu().numpy()
            ranked = np.argsort(-scores)
            pos_rank = int(np.where(ranked == 0)[0][0])
            hr_list.append(1.0 if pos_rank < TOP_K else 0.0)
            ndcg_list.append(1.0 / np.log2(pos_rank + 2) if pos_rank < TOP_K else 0.0)

    return {
        f"NDCG@{TOP_K}": np.mean(ndcg_list) if ndcg_list else 0.0,
        f"HR@{TOP_K}": np.mean(hr_list) if hr_list else 0.0
    }


print("Evaluation functions ready.")

Evaluation functions ready.


## 14. Run Single Fold

In [15]:
def run_fold(fold_tag, train_inters, test_inters, do_ranking=True):
    """Full pipeline for one fold: features -> prior -> model -> train -> eval."""
    print(f"\n{'=' * 65}")
    print(f"  [{fold_tag}]: {len(train_inters):,} train / {len(test_inters):,} test")
    print(f"{'=' * 65}")

    # 1. Compute leak-free features
    feats = compute_fold_features(train_inters, fold_tag)
    gm = feats["global_mean"]
    print(f"  Global mean: {gm:.3f}")

    # 2. Pre-train item prior
    print("  Pre-training item prior...")
    item_prior = pretrain_item_prior(train_inters, feats)

    # 3. Build datasets
    i_train_avg = feats["item_train_avg"]
    train_ds = RecDataset(train_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    test_ds = RecDataset(test_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_CPU_WORKERS, pin_memory=PIN_MEMORY,
                 prefetch_factor=4 if NUM_CPU_WORKERS > 0 else None,
                 persistent_workers=True if NUM_CPU_WORKERS > 0 else False)
    train_dl = DataLoader(train_ds, shuffle=True, **dl_kw)
    test_dl = DataLoader(test_ds, shuffle=False, **dl_kw)

    # 4. Build model
    model = BESTRecV3(num_users, num_items, gm,
                       feats["user_bias"], feats["item_bias"], item_prior).to(device)
    raw_model = model
    try:
        if hasattr(torch, "compile"):
            model = torch.compile(model, mode="reduce-overhead")
            print("  Model compiled")
    except Exception as e:
        print(f"  Compile skipped: {e}")

    # 5. Optimiser + scheduler
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_dl)
    warmup_steps = int(WARMUP_FRAC * total_steps)
    def lr_fn(step):
        if step < warmup_steps: return step / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * prog))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_fn)
    scaler = GradScaler(enabled=USE_AMP)

    # 6. Training with curriculum DropoutNet
    best_mae, best_state, pat = float("inf"), None, 0
    for epoch in range(EPOCHS):
        t0 = time.time()

        # Set curriculum dropout rate
        dp = get_dropout_p(epoch, EPOCHS)
        raw_model.current_dropout_p = dp

        loss = train_one_epoch(model, optimizer, scheduler, scaler, train_dl, feats)
        mae, rmse = evaluate_rating(model, test_dl)
        dt = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]
        mk = ""
        if mae < best_mae:
            best_mae = mae
            best_state = copy.deepcopy(raw_model.state_dict())
            pat = 0; mk = " *"
        else:
            pat += 1
        print(f"  Ep {epoch+1:2d} | loss={loss:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f} | dp={dp:.2f} | lr={lr_now:.1e} | {dt:.0f}s{mk}")
        if pat >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}"); break

    # 7. Restore best and evaluate
    raw_model.load_state_dict(best_state)
    final_mae, final_rmse = evaluate_rating(model, test_dl)
    results = {"mae": final_mae, "rmse": final_rmse}
    print(f"  Best: MAE={final_mae:.4f}, RMSE={final_rmse:.4f}")

    if do_ranking:
        rank = evaluate_ranking(model, test_inters, feats, item_title_embeds,
                                item_numeric, item_raw_avg, feats["item_train_avg"])
        results.update(rank)
        for k, v in rank.items(): print(f"  {k}: {v:.4f}")

    # Cleanup
    del model, raw_model, optimizer, scheduler, scaler, train_dl, test_dl, item_prior
    torch.cuda.empty_cache(); gc.collect()
    return results

print("run_fold ready.")

run_fold ready.


## 15. Experiment 1: Warm Evaluation (Standard Random Split)

**This is the main result table for the paper.**

Standard 5-fold CV with random split on interactions. Users/items CAN appear in both train and test.
This is what most published papers report — it gives the best numbers and is the standard protocol.

In [ ]:
print("=" * 65)
print(f"  EXPERIMENT 1: WARM EVALUATION (Standard KFold) - {DATASET.upper()}")
print("=" * 65)

indices = np.arange(len(interactions))
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)

warm_results = []
for fold_idx, (tr, te) in enumerate(kf.split(indices)):
    result = run_fold(
        f"warm_f{fold_idx}",
        [interactions[i] for i in tr],
        [interactions[i] for i in te],
        do_ranking=True
    )
    warm_results.append(result)

print("\n" + "=" * 65)
print("WARM RESULTS (main paper table)")
print("=" * 65)
for m in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[m] for r in warm_results if m in r]
    if vals:
        print(f"  {m:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

  EXPERIMENT 1: WARM EVALUATION (Standard KFold) - BEAUTY

  [warm_f0]: 561,222 train / 140,306 test
  Computing: item_svd.pt...
  Saved: item_svd.pt
  Computing: user_svd.pt...
  Saved: user_svd.pt
  Computing: user_text.pkl...


Batches:   0%|          | 0/1088 [00:00<?, ?it/s]

  Saved: user_text.pkl (4856.8 MB)
  Global mean: 3.960
  Pre-training item prior...
  Item prior trained: val MSE = 0.9485 (RMSE = 0.9739)
  Model compiled


  Train:   0%|          | 0/138 [00:00<?, ?it/s]

W0409 10:18:17.626000 3079229 torch/_inductor/utils.py:1731] [1/0] Not enough SMs to use max_autotune_gemm mode


...

  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Best: MAE=1.0772, RMSE=1.5967


  Ranking:   0%|          | 0/2000 [00:00<?, ?it/s]

  NDCG@10: 0.0833
  HR@10: 0.1296

  [warm_f4]: 561,223 train / 140,305 test
  Computing: item_svd.pt...
  Saved: item_svd.pt
  Computing: user_svd.pt...
  Saved: user_svd.pt
  Computing: user_text.pkl...


Batches:   0%|          | 0/1088 [00:00<?, ?it/s]

## 16. Experiment 2: Cold-Start Evaluation

Two protocols:
1. **GroupKFold** (strict): entire users held out, no user signal at test time
2. **Cold-user holdout**: users with <=3 interactions entirely removed from training
3. **Cold-item holdout**: items with <=5 interactions entirely removed from training

This addresses the reviewer's concern about cold-start evaluation.

In [ ]:
print("=" * 65)
print(f"  EXPERIMENT 2: COLD-START EVALUATION - {DATASET.upper()}")
print("=" * 65)

# --- 2a: GroupKFold (strict cold-user) ---
print("\n--- GroupKFold (strict: entire users held out) ---")
user_ids_arr = np.array([i["user_id"] for i in interactions])
gkf = GroupKFold(n_splits=NUM_FOLDS)

cold_gkf_results = []
for fold_idx, (tr, te) in enumerate(gkf.split(indices, groups=user_ids_arr)):
    result = run_fold(
        f"cold_gkf_f{fold_idx}",
        [interactions[i] for i in tr],
        [interactions[i] for i in te],
        do_ranking=True
    )
    cold_gkf_results.append(result)

print("\nGroupKFold Results:")
for m in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[m] for r in cold_gkf_results if m in r]
    if vals:
        print(f"  {m:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

# --- 2b: Cold-user holdout ---
def split_cold(inters, key, thr):
    counts = defaultdict(int)
    for i in inters: counts[i[key]] += 1
    cold = {k for k, c in counts.items() if c <= thr}
    return [i for i in inters if i[key] not in cold], [i for i in inters if i[key] in cold]

print("\n--- Cold-user holdout ---")
cu_tr, cu_te = split_cold(interactions, "user_id", COLD_USER_THRESHOLD)
print(f"  Cold users: {len(cu_te):,} test interactions")
cu_res = run_fold("cold_user", cu_tr, cu_te, do_ranking=False) if len(cu_te) >= 10 else None

print("\n--- Cold-item holdout ---")
ci_tr, ci_te = split_cold(interactions, "item_id", COLD_ITEM_THRESHOLD)
print(f"  Cold items: {len(ci_te):,} test interactions")
ci_res = run_fold("cold_item", ci_tr, ci_te, do_ranking=False) if len(ci_te) >= 10 else None

print("\n" + "=" * 65)
print("COLD-START SUMMARY")
print("=" * 65)
warm_mae = np.mean([r["mae"] for r in warm_results])
gkf_mae = np.mean([r["mae"] for r in cold_gkf_results])
print(f"  Warm MAE:       {warm_mae:.4f}")
print(f"  GroupKFold MAE: {gkf_mae:.4f}  (degradation: {(gkf_mae-warm_mae)/warm_mae*100:.1f}%)")
if cu_res: print(f"  Cold-user MAE:  {cu_res['mae']:.4f}")
if ci_res: print(f"  Cold-item MAE:  {ci_res['mae']:.4f}")

## 17. Experiment 3: Fair Baselines (NeuMF + SVD)

Run baselines on the **exact same splits** as v3.0 for fair comparison.

In [ ]:
class NeuMF(nn.Module):
    """Neural Matrix Factorization baseline."""
    def __init__(self, n_users, n_items, emb_dim=64, hidden=128):
        super().__init__()
        # GMF
        self.u_emb_gmf = nn.Embedding(n_users, emb_dim)
        self.i_emb_gmf = nn.Embedding(n_items, emb_dim)
        # MLP
        self.u_emb_mlp = nn.Embedding(n_users, emb_dim)
        self.i_emb_mlp = nn.Embedding(n_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.1),
        )
        self.out = nn.Linear(emb_dim + hidden // 2, 1)
        nn.init.xavier_uniform_(self.u_emb_gmf.weight)
        nn.init.xavier_uniform_(self.i_emb_gmf.weight)
        nn.init.xavier_uniform_(self.u_emb_mlp.weight)
        nn.init.xavier_uniform_(self.i_emb_mlp.weight)

    def forward(self, user_ids, item_ids):
        gmf = self.u_emb_gmf(user_ids) * self.i_emb_gmf(item_ids)
        mlp_in = torch.cat([self.u_emb_mlp(user_ids), self.i_emb_mlp(item_ids)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        x = torch.cat([gmf, mlp_out], dim=-1)
        return torch.clamp(self.out(x).squeeze(-1) + 3.0, 1.0, 5.0)  # bias toward mean


def train_neumf(train_inters, test_inters, n_users, n_items, epochs=30):
    """Train and evaluate NeuMF baseline."""
    model = NeuMF(n_users, n_items).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    # Simple dataloader
    train_u = torch.tensor([i["user_id"] for i in train_inters], dtype=torch.long)
    train_i = torch.tensor([i["item_id"] for i in train_inters], dtype=torch.long)
    train_r = torch.tensor([i["rating"] for i in train_inters], dtype=torch.float32)
    test_u = torch.tensor([i["user_id"] for i in test_inters], dtype=torch.long)
    test_i = torch.tensor([i["item_id"] for i in test_inters], dtype=torch.long)
    test_r = torch.tensor([i["rating"] for i in test_inters], dtype=torch.float32)

    best_mae, best_state, pat = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(train_r))
        total_loss, n = 0.0, 0
        for start in range(0, len(train_r), BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            pred = model(train_u[idx].to(device), train_i[idx].to(device))
            loss = F.mse_loss(pred, train_r[idx].to(device))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total_loss += loss.item() * len(idx); n += len(idx)

        model.eval()
        with torch.no_grad():
            all_p = []
            for start in range(0, len(test_r), BATCH_SIZE):
                p = model(test_u[start:start+BATCH_SIZE].to(device),
                          test_i[start:start+BATCH_SIZE].to(device))
                all_p.append(p.cpu().numpy())
            pred_all = np.concatenate(all_p)
        mae = mean_absolute_error(test_r.numpy(), pred_all)
        if mae < best_mae:
            best_mae = mae; best_state = copy.deepcopy(model.state_dict()); pat = 0
        else: pat += 1
        if pat >= 8: break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        all_p = []
        for start in range(0, len(test_r), BATCH_SIZE):
            p = model(test_u[start:start+BATCH_SIZE].to(device),
                      test_i[start:start+BATCH_SIZE].to(device))
            all_p.append(p.cpu().numpy())
    pred_all = np.concatenate(all_p)
    mae = mean_absolute_error(test_r.numpy(), pred_all)
    rmse = np.sqrt(mean_squared_error(test_r.numpy(), pred_all))
    del model; torch.cuda.empty_cache()
    return {"mae": mae, "rmse": rmse}


def svd_baseline(train_inters, test_inters, k=128):
    """SVD + mean bias baseline."""
    # Compute biases
    gs, gn = 0.0, 0
    us, uc = defaultdict(float), defaultdict(int)
    ist, ic = defaultdict(float), defaultdict(int)
    for i in train_inters:
        r = i["rating"]; gs += r; gn += 1
        us[i["user_id"]] += r; uc[i["user_id"]] += 1
        ist[i["item_id"]] += r; ic[i["item_id"]] += 1
    gm = gs / gn

    # SVD on residuals
    rows, cols, vals = [], [], []
    for i in train_inters:
        ub = (us[i["user_id"]] / uc[i["user_id"]]) - gm if uc[i["user_id"]] > 0 else 0
        ib = (ist[i["item_id"]] / ic[i["item_id"]]) - gm if ic[i["item_id"]] > 0 else 0
        residual = i["rating"] - gm - ub - ib
        rows.append(i["user_id"]); cols.append(i["item_id"]); vals.append(residual)

    mat = csr_matrix((vals, (rows, cols)), shape=(num_users, num_items))
    actual_k = min(k, min(num_users, num_items) - 1, len(set(rows)) - 1)
    if actual_k < 1:
        # Fallback to bias-only
        preds = []
        for i in test_inters:
            ub = (us[i["user_id"]] / uc[i["user_id"]]) - gm if uc[i["user_id"]] > 0 else 0
            ib = (ist[i["item_id"]] / ic[i["item_id"]]) - gm if ic[i["item_id"]] > 0 else 0
            preds.append(np.clip(gm + ub + ib, 1, 5))
    else:
        svd = TruncatedSVD(n_components=actual_k, random_state=SEED)
        U = svd.fit_transform(mat)  # (users, k)
        V = svd.components_  # (k, items)

        preds = []
        for i in test_inters:
            ub = (us[i["user_id"]] / uc[i["user_id"]]) - gm if uc[i["user_id"]] > 0 else 0
            ib = (ist[i["item_id"]] / ic[i["item_id"]]) - gm if ic[i["item_id"]] > 0 else 0
            svd_pred = U[i["user_id"]] @ V[:, i["item_id"]] if uc[i["user_id"]] > 0 else 0
            preds.append(np.clip(gm + ub + ib + svd_pred, 1, 5))

    targets = [i["rating"] for i in test_inters]
    return {
        "mae": mean_absolute_error(targets, preds),
        "rmse": np.sqrt(mean_squared_error(targets, preds))
    }


# Run baselines on warm fold 0
print("=" * 65)
print(f"  EXPERIMENT 3: BASELINES (same splits) - {DATASET.upper()}")
print("=" * 65)

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
splits = list(kf.split(indices))
tr0, te0 = splits[0]
train0 = [interactions[i] for i in tr0]
test0 = [interactions[i] for i in te0]

# Global mean baseline
gm_preds = np.full(len(te0), np.mean([i["rating"] for i in train0]))
gm_mae = mean_absolute_error([interactions[i]["rating"] for i in te0], gm_preds)
print(f"\n  Global Mean:  MAE = {gm_mae:.4f}")

# SVD baseline
svd_res = svd_baseline(train0, test0)
print(f"  SVD (k=128):  MAE = {svd_res['mae']:.4f}, RMSE = {svd_res['rmse']:.4f}")

# NeuMF baseline
neumf_res = train_neumf(train0, test0, num_users, num_items)
print(f"  NeuMF:        MAE = {neumf_res['mae']:.4f}, RMSE = {neumf_res['rmse']:.4f}")

# v3.0 fold 0 result
v3_f0 = warm_results[0]
print(f"  BEST-Rec v3:  MAE = {v3_f0['mae']:.4f}, RMSE = {v3_f0['rmse']:.4f}")
if svd_res['mae'] > 0:
    imp = (svd_res['mae'] - v3_f0['mae']) / svd_res['mae'] * 100
    print(f"  Improvement over SVD: {imp:.1f}%")

# Also run baselines on GroupKFold fold 0
gkf_splits = list(GroupKFold(n_splits=NUM_FOLDS).split(indices, groups=user_ids_arr))
gtr0, gte0 = gkf_splits[0]
gtrain0 = [interactions[i] for i in gtr0]
gtest0 = [interactions[i] for i in gte0]

print(f"\n  --- Cold (GroupKFold fold 0) ---")
svd_cold = svd_baseline(gtrain0, gtest0)
print(f"  SVD (cold):   MAE = {svd_cold['mae']:.4f}, RMSE = {svd_cold['rmse']:.4f}")
neumf_cold = train_neumf(gtrain0, gtest0, num_users, num_items)
print(f"  NeuMF (cold): MAE = {neumf_cold['mae']:.4f}, RMSE = {neumf_cold['rmse']:.4f}")
v3_cold_f0 = cold_gkf_results[0]
print(f"  v3 (cold):    MAE = {v3_cold_f0['mae']:.4f}, RMSE = {v3_cold_f0['rmse']:.4f}")

## 18. Experiment 4: Ablation Study

Test each component's contribution on warm fold 0. Variants:
1. **Full v3.0** (all components)
2. **No DropoutNet** (curriculum dropout p=0 always)
3. **No bridge** (remove content-CF alignment loss)
4. **No cross-attention** (skip CA, use concat of user_pool + item_pool)
5. **No item prior** (remove pre-trained item prior, use bias only for cold)
6. **No BPR** (remove ranking loss)

In [ ]:
def run_ablation_fold(variant_name, train_inters, test_inters, modifications):
    """Run a single ablation variant."""
    print(f"\n  --- Ablation: {variant_name} ---")

    # Save original config
    orig = {}
    for key, val in modifications.items():
        orig[key] = globals()[key]
        globals()[key] = val

    try:
        result = run_fold(f"abl_{variant_name}", train_inters, test_inters, do_ranking=False)
    finally:
        # Restore config
        for key, val in orig.items():
            globals()[key] = val

    return result


print("=" * 65)
print(f"  EXPERIMENT 4: ABLATION STUDY - {DATASET.upper()}")
print("=" * 65)

# Use warm fold 0
abl_tr = [interactions[i] for i in splits[0][0]]
abl_te = [interactions[i] for i in splits[0][1]]

ablation_results = {}

# Full model (already computed)
ablation_results["Full v3.0"] = warm_results[0]
print(f"  Full v3.0: MAE = {warm_results[0]['mae']:.4f}")

# No DropoutNet
ablation_results["No DropoutNet"] = run_ablation_fold(
    "no_dropout", abl_tr, abl_te, {"DROPOUT_TARGET": 0.0})

# No BPR
ablation_results["No BPR"] = run_ablation_fold(
    "no_bpr", abl_tr, abl_te, {"LAMBDA_BPR": 0.0})

# No bridge
ablation_results["No Bridge"] = run_ablation_fold(
    "no_bridge", abl_tr, abl_te, {"LAMBDA_BRIDGE": 0.0})

# No contrastive
ablation_results["No Contrastive"] = run_ablation_fold(
    "no_cl", abl_tr, abl_te, {"LAMBDA_CL": 0.0})

print("\n" + "=" * 65)
print("ABLATION RESULTS")
print("=" * 65)
for name, res in ablation_results.items():
    print(f"  {name:>20s}: MAE = {res['mae']:.4f}, RMSE = {res['rmse']:.4f}")

## 19. Results Summary & Export

In [ ]:
def jsonify(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, dict): return {k: jsonify(v) for k, v in o.items()}
    if isinstance(o, list): return [jsonify(v) for v in o]
    return o


# Full summary
print("\n" + "#" * 70)
print(f"  BEST-Rec v3.0 COMPLETE RESULTS: {DATASET.upper()}")
print("#" * 70)

print("\n=== WARM (main paper table) ===")
for m in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[m] for r in warm_results if m in r]
    if vals:
        mean, std = np.mean(vals), np.std(vals)
        ci95 = 1.96 * std / np.sqrt(len(vals))
        print(f"  {m:>10s}: {mean:.4f} +/- {std:.4f} (95% CI: [{mean-ci95:.4f}, {mean+ci95:.4f}])")

print("\n=== COLD (GroupKFold) ===")
for m in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[m] for r in cold_gkf_results if m in r]
    if vals:
        print(f"  {m:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

print("\n=== COLD-START HOLDOUT ===")
if cu_res: print(f"  Cold-user: MAE={cu_res['mae']:.4f}, RMSE={cu_res['rmse']:.4f}")
if ci_res: print(f"  Cold-item: MAE={ci_res['mae']:.4f}, RMSE={ci_res['rmse']:.4f}")

print("\n=== BASELINES (fold 0) ===")
print(f"  Global Mean: MAE={gm_mae:.4f}")
print(f"  SVD:         MAE={svd_res['mae']:.4f}, RMSE={svd_res['rmse']:.4f}")
print(f"  NeuMF:       MAE={neumf_res['mae']:.4f}, RMSE={neumf_res['rmse']:.4f}")

print("\n=== ABLATION (fold 0) ===")
for name, res in ablation_results.items():
    print(f"  {name:>20s}: MAE={res['mae']:.4f}, RMSE={res['rmse']:.4f}")

# Save to JSON
all_results = jsonify({
    "dataset": DATASET,
    "version": "v3.0",
    "warm": warm_results,
    "cold_groupkfold": cold_gkf_results,
    "cold_user": cu_res,
    "cold_item": ci_res,
    "baselines": {
        "global_mean": {"mae": float(gm_mae)},
        "svd": svd_res,
        "neumf": neumf_res,
        "svd_cold": svd_cold,
        "neumf_cold": neumf_cold,
    },
    "ablation": {k: v for k, v in ablation_results.items()},
})

out_path = os.path.join(V3_CACHE, "v3_results.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to {out_path}")
print("\nDone! Share this JSON to proceed with the paper.")